## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-11-21-04-00 +0000,wsj,U.S. Natural Gas Futures Settle Lower After Gains,https://www.wsj.com/finance/commodities-future...
1,2026-11-11-21-03-00 +0000,wsj,Opinion | Let the 2028 Presidential Campaign B...,https://www.wsj.com/opinion/let-the-2028-presi...
2,2026-11-11-21-01-56 +0000,nyt,How Democrats Invested Online to Overcome Thei...,https://www.nytimes.com/2026/11/11/us/politics...
3,2026-11-11-21-00-59 +0000,nypost,California mom taken off life support after hu...,https://nypost.com/2026/11/11/us-news/vacavill...
4,2026-11-11-21-00-00 +0000,wapo,"After Democrats’ midterm wins, Palantir CEO Al...",https://www.washingtonpost.com/technology/2026...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2630/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
68,trump,36
105,house,20
125,war,18
539,nvidia,17
535,huang,14


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
15,2026-11-11-20-40-00 +0000,wsj,"Democrats, fresh off sweeping midterm wins, ar...",https://www.wsj.com/politics/national-security...,123
215,2026-11-11-11-53-00 +0000,wsj,House Republicans saved Pete Hegseth and picke...,https://www.wsj.com/politics/policy/house-gop-...,122
166,2026-11-11-14-49-48 +0000,nypost,White House blasts Democrats planning Trump ad...,https://nypost.com/2026/11/11/us-news/white-ho...,108
147,2026-11-11-15-55-57 +0000,latimes,White House warns of criminal inquiries into H...,https://www.latimes.com/politics/story/2026-11...,108
352,2026-11-11-00-07-28 +0000,nypost,Chinese ‘spy’ Alice Huang met Nvidia CEO Jense...,https://nypost.com/2026/11/10/business/alice-h...,106


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
15,123,2026-11-11-20-40-00 +0000,wsj,"Democrats, fresh off sweeping midterm wins, ar...",https://www.wsj.com/politics/national-security...
352,106,2026-11-11-00-07-28 +0000,nypost,Chinese ‘spy’ Alice Huang met Nvidia CEO Jense...,https://nypost.com/2026/11/10/business/alice-h...
321,94,2026-11-11-02-15-00 +0000,wsj,House Republicans elected Rep. Jim Jordan of O...,https://www.wsj.com/politics/jim-jordan-wins-s...
342,72,2026-11-11-00-34-00 +0000,wsj,Defense Secretary Pete Hegseth gave a Pentagon...,https://www.wsj.com/politics/national-security...
171,66,2026-11-11-14-40-48 +0000,nypost,Joe Biden claims he would have resigned if he ...,https://nypost.com/2026/11/11/us-news/joe-bide...
323,51,2026-11-11-02-09-00 +0000,wsj,The Kennedy family is joining the efforts to b...,https://www.wsj.com/politics/policy/kennedy-fa...
345,49,2026-11-11-00-29-45 +0000,latimes,Meghan Markle's father Thomas Markle Sr. cut o...,https://www.latimes.com/entertainment-arts/sto...
332,49,2026-11-11-01-34-46 +0000,nypost,California Dems plotting sneaky mileage tax un...,https://nypost.com/2026/11/10/us-news/xavier-b...
283,47,2026-11-11-08-21-53 +0000,wsj,"Stock Market Today: CoreWeave Shares Jump, Re-...",https://www.wsj.com/livecoverage/stock-market-...
76,44,2026-11-11-18-20-00 +0000,chitri,‘You look up and there it is’: Amazon Prime Ai...,https://www.chicagotribune.com/2026/11/11/amaz...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
